
## 📦 Install Dependencies

In [ ]:
!pip install accelerate bitsandbytes datasets huggingface_hub  peft scikit-learn transformers trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.2/517.2 kB 43.0 MB/s eta 0:00:00


## 📚 Libraries

In [ ]:
from collections import Counter
from datasets import concatenate_datasets, load_dataset
from google.colab import userdata
from huggingface_hub import login, notebook_login
import numpy as np
import os
from peft import LoraConfig
import random
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
)
from torch.nn import CrossEntropyLoss
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline, TrainingArguments
from trl import SFTTrainer
import torch

## 🔐 Login to Hugging Face Hub

In [ ]:
hf_token = os.environ.get('HF_Token') or userdata.get('HF_Token')

if hf_token:
    login(token=hf_token)
    print("HuggingFace login successful.")
else:
    print("HuggingFace token not found. Please set the HF_TOKEN environment variable or store it in Colab secrets.")



HuggingFace login successful.


## 📥 Load dair-ai/emotion Dataset

In [ ]:
dataset = load_dataset("dair-ai/emotion")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
# Inspect dataset
print(f"\ndataset['train'] head:")
print(dataset["train"][:5])

print("\ndataset['train'] column names:")
print(dataset["train"].column_names)


dataset['train'] head:
{'text': ['i didnt feel humiliated', 'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake', 'im grabbing a minute to post i feel greedy wrong', 'i am ever feeling nostalgic about the fireplace i will know that it is still on the property', 'i am feeling grouchy'], 'label': [0, 0, 3, 2, 3]}

dataset['train'] column names:
['text', 'label']


🫥 Convert Answer Integers to Text

In [ ]:
emotion_map = {
  0: "sadness",
  1: "joy",
  2: "love",
  3: "anger",
  4: "fear",
  5: "surprise"
}

def convert_label(dataset):
    dataset["emotion"] = emotion_map[dataset["label"]]
    return dataset

dataset['train'] = dataset['train'].map(convert_label)
dataset['test'] = dataset['train'].map(convert_label)

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

In [ ]:
# Inspect dataset
print(f"\ndataset['train'] head:")
print(dataset["train"][:5])

print("\ndataset['train'] column names:")
print(dataset["train"].column_names)




dataset['train'] head:
{'text': ['i didnt feel humiliated', 'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake', 'im grabbing a minute to post i feel greedy wrong', 'i am ever feeling nostalgic about the fireplace i will know that it is still on the property', 'i am feeling grouchy'], 'label': [0, 0, 3, 2, 3], 'emotion': ['sadness', 'sadness', 'anger', 'love', 'anger']}

dataset['train'] column names:
['text', 'label', 'emotion']


⚖️ Inspect Dataset Balance

In [ ]:
label_counts = Counter(dataset["train"]["label"])
print(label_counts)

Counter({1: 5362, 0: 4666, 3: 2159, 4: 1937, 2: 1304, 5: 572})


The dataset is unbalanced.  And so I add class weights (loss weights) so that more rare classes have more weight

In [ ]:
# Compute class weights = inverse frequency
counts = torch.tensor([4666, 5362, 1304, 2159, 1937, 572], dtype=torch.float)
weights = 1.0 / counts
weights = weights / weights.sum()  # normalize

loss_fn = CrossEntropyLoss(weight=weights)

In [ ]:
## 🧠 Define the options + token IDs

In [ ]:
OPTIONS = ["sadness", "joy", "love", "anger", "fear", "surprise"]

In [ ]:
print(dataset["train"][0])

{'text': 'i didnt feel humiliated', 'label': 0, 'emotion': 'sadness'}


## 🧩 Create Prompts from Tabular Data



In [ ]:
def row_to_prompt(example):
    question = example["text"]

    # Build bullet list
    options_text = "\n".join([f"- '{opt}'" for opt in OPTIONS])

    prompt = (
        "You are an expert at classifying emotions from text.\n\n"
        f"Question:\n{question}\n\n"
        f"Options:\n{options_text}\n\n"
        "Answer with only one choice from the options."
    )

    target = example["emotion"]  # already mapped to string label

    return {"prompt": prompt, "target": target}

train_ds = dataset["train"].map(row_to_prompt)
test_ds = dataset["test"].map(row_to_prompt)

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

In [ ]:
# Examine row
print(row_to_prompt(test_ds[0]))

{'prompt': "You are an expert at classifying emotions from text.\n\nQuestion:\ni didnt feel humiliated\n\nOptions:\n- 'sadness'\n- 'joy'\n- 'love'\n- 'anger'\n- 'fear'\n- 'surprise'\n\nAnswer with only one choice from the options.", 'target': 'sadness'}


## 📊 Baseline Model Evaluation (Before Fine-Tuning)

In [ ]:
def predict_label_baseline(prompt: str) -> str:
    """
    Baseline model: choose from:
      - "sadness"
      - "joy"
      - "love"
      - "anger"
      - "fear"
      - "surprise"
    using logits from the base (unfine-tuned) model.
    """
    msg = [{"role": "user", "content": prompt}]
    prompt_text = tok.apply_chat_template(
        msg,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tok(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)           # base model
        logits = outputs.logits             # [batch, seq_len, vocab]

    last_logits = logits[0, -1]             # [vocab] logits for next token

    # Score each option by the logit of its (last) token
    scores = [last_logits[token_id].item() for token_id in answer_token_ids]
    best_idx = int(np.argmax(scores))

    # Return the emotion string
    return OPTIONS[best_idx]

🧪 Reproducibility

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

🧠 Load base model + tokenizer

In [ ]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

# Tokenizer
tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    use_fast=True,
)

if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# Map each emotion option to the id of its *last* token
answer_token_ids = [
    tok(opt, add_special_tokens=False).input_ids[-1]
    for opt in OPTIONS
]

# Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",          # GPU if available
    torch_dtype=torch.bfloat16, # efficient on modern GPUs
    trust_remote_code=True,
    token=hf_token,
)

# Optional pipeline (not strictly needed for baseline)
gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tok,
    device_map="auto",
    dtype=torch.bfloat16,
    max_new_tokens=1,
    do_sample=False,
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


📊 Baseline evaluation

In [ ]:
N_PREVIEW = min(200, len(test_ds))

y_true = [test_ds[i]["target"] for i in range(N_PREVIEW)]                  # gold emotions
y_pred = [predict_label_baseline(test_ds[i]["prompt"]) for i in range(N_PREVIEW)]

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="macro",
    zero_division=0,
)

print(f"\nBaseline (N={N_PREVIEW})")
print(f"  Acc={acc:.3f}  Prec={prec:.3f}  Rec={rec:.3f}  F1={f1:.3f}\n")

print("Classification report:")
print(classification_report(y_true, y_pred, labels=OPTIONS, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=OPTIONS)
print("\nConfusion matrix (rows=true, cols=pred):")
print("   " + "  ".join(OPTIONS))
for i, row in enumerate(cm):
    print(OPTIONS[i], row)


Baseline (N=200)
  Acc=0.390  Prec=0.217  Rec=0.352  F1=0.231

Classification report:
              precision    recall  f1-score   support

     sadness       0.00      0.00      0.00        47
         joy       0.82      0.54      0.65        76
        love       0.23      0.67      0.34        12
       anger       0.25      0.91      0.39        32
        fear       0.00      0.00      0.00        22
    surprise       0.00      0.00      0.00        11

    accuracy                           0.39       200
   macro avg       0.22      0.35      0.23       200
weighted avg       0.37      0.39      0.33       200


Confusion matrix (rows=true, cols=pred):
   sadness  joy  love  anger  fear  surprise
sadness [ 0  1  8 38  0  0]
joy [ 0 41 13 22  0  0]
love [0 1 8 3 0 0]
anger [ 0  1  2 29  0  0]
fear [ 0  1  2 19  0  0]
surprise [0 5 2 4 0 0]


## ⚙️ Install + Reload Model in 4-bit (QLoRA-Ready)


In [37]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

# 4-bit quantization config (QLoRA-style)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Tokenizer
tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    use_fast=True,
)

if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# Clear cache just in case
torch.cuda.empty_cache()

# 🚀 Load model with 4-bit config
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0}, # force gpu
    trust_remote_code=True,
    token=hf_token,
)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## 🛠️ QLoRA config + Trainer

In [51]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

def formatting_func(batch):
    prompts = batch["prompt"]
    targets = batch["target"]
    outputs = []

    for p, t in zip(prompts, targets):
        outputs.append(p + "\nAnswer: " + t)

    return {"text": outputs}

# Convert datasets
train_for_sft = train_ds.map(
    formatting_func,
    batched=True,
    remove_columns=train_ds.column_names,
)

eval_for_sft = test_ds.select(range(500)).map(
    formatting_func,
    batched=True,
    remove_columns=test_ds.column_names,
)

# ---------------------------
# 🎯 Smol Training Arguments
# ---------------------------
output_dir = "emotion-llama-qlora"

args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    num_train_epochs=2,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=20,
    logging_first_step=True,
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    bf16=True,
    report_to="none",
    gradient_checkpointing=True,
)


# ---------------------------
# 🚂 QLoRA SFTTrainer
# ---------------------------
trainer = SFTTrainer(
    model=model,
    peft_config=lora_config,
    train_dataset=train_for_sft,
    eval_dataset=eval_for_sft,
    args=args,
)

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/16000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/16000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/16000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

## 🚀 Fine-Tune Model with QLoRA


In [ ]:
trainer.train()

Step,Training Loss
1,3.001600
20,2.935600
40,2.071700
60,1.186000
80,1.008100
100,1.015200
120,0.927400
140,0.971000
160,0.916200
180,0.923500


## 💾 Save the LoRA adapter and push to HuggingFace Hub

In [ ]:
adapter_dir = "emotion-llama-qlora"
trainer.model.save_pretrained(adapter_dir)
tok.save_pretrained(adapter_dir)
print("Saved:", adapter_dir)
trainer.model.push_to_hub("david125tran/emotion-llama-qlora")
tok.push_to_hub("david125tran/emotion-llama-qlora")